# PP-OCRv4 Mobile Det 訓練（Colab GPU）

本筆記本用於在 Google Colab 上訓練 PP-OCRv4 mobile **文字偵測 (det)** 模型。

**GitHub Repo:** `https://github.com/install88/trainingOcr`

**流程：**
1. Clone repo + 安裝環境
2. 上傳標注好的資料集（圖片 + label txt）
3. 下載預訓練模型
4. 訓練 det 模型
5. 匯出推論模型 → 轉 ONNX
6. 下載訓練好的模型

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. Clone Repo + 安裝環境

In [ ]:
import os
os.chdir("/content")

# Clone 專案 repo
if not os.path.exists("trainingOcr"):
    !git clone https://github.com/install88/trainingOcr.git

# Clone PaddleOCR 訓練框架
if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

# 安裝 PaddlePaddle GPU 版（Colab 通常是 CUDA 12.x）
# 如果 Colab 用的是 CUDA 11.8，把 cu126 改成 cu118
!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# 安裝其他依賴
!pip install paddleocr paddle2onnx onnxruntime shapely pyclipper imgaug lmdb

In [ ]:
# 驗證安裝
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.is_compiled_with_cuda()}")
print(f"GPU count: {paddle.device.cuda.device_count()}")

In [ ]:
import os
os.chdir("/content")

if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

os.chdir("/content/PaddleOCR")
!pwd

In [ ]:
# 方法 A：從 Google Drive 載入（推薦）
from google.colab import drive
drive.mount('/content/drive')

# 假設資料集 zip 放在 Google Drive 的 My Drive/ocr_project/ 下
DRIVE_DATASET_PATH = "/content/drive/MyDrive/ocr_project/det_dataset.zip"

!mkdir -p /content/dataset/det
!unzip -o "{DRIVE_DATASET_PATH}" -d /content/dataset/det/

## 3. 上傳資料集

repo 已 clone 到 `/content/trainingOcr/`，但圖片和標注不在 repo 裡（太大）。

你需要上傳 `det_dataset.zip`（由本機 `split_dataset.py` 產生的 `dataset/det/` 打包）：
```
det_dataset.zip
├── train/           ← 訓練圖片
├── val/             ← 驗證圖片
├── train_label.txt  ← 訓練標注
└── val_label.txt    ← 驗證標注
```

In [ ]:
# 檢查資料集
!echo "=== 資料集結構 ==="
!ls -la /content/dataset/det/
!echo ""
!echo "=== 訓練集圖片數量 ==="
!ls /content/dataset/det/train/ | wc -l
!echo "=== 驗證集圖片數量 ==="
!ls /content/dataset/det/val/ | wc -l
!echo ""
!echo "=== train_label.txt 前 3 行 ==="
!head -3 /content/dataset/det/train_label.txt
!echo ""
!echo "=== val_label.txt 前 3 行 ==="
!head -3 /content/dataset/det/val_label.txt

## 4. 下載預訓練模型

In [ ]:
!mkdir -p /content/pretrained_models

# 下載 PP-OCRv4 det Mobile 模型（student，PPLCNetV3 backbone）
# ★ 必須用 student 模型，不能用 train（那是 Server 版 ResNet50，架構不同）
!wget -nc -P /content/pretrained_models/ 
    https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_student.tar

!cd /content/pretrained_models && tar xf ch_PP-OCRv4_det_student.tar

print("預訓練模型下載完成！")
!ls -la /content/pretrained_models/ch_PP-OCRv4_det_student/

PRETRAIN = "/content/pretrained_models/ch_PP-OCRv4_det_student/best_accuracy"


## 5. 使用 repo 的訓練設定檔

直接用 repo 裡的 config，只需覆寫路徑為 Colab 環境的路徑。

In [ ]:
# 確認 repo config 存在
REPO_DIR = "/content/trainingOcr"
CONFIG_PATH = f"{REPO_DIR}/configs/det/PP-OCRv4_mobile_det_finetune.yml"
!cat {CONFIG_PATH} | head -20
print(f"\n設定檔路徑：{CONFIG_PATH}")

## 6. Baseline 評估（訓練前）

先用 **官方 PP-OCRv4 det 模型**（還沒 fine-tune）在我們自己的驗證集跑一次，記下 hmean / precision / recall 作為 baseline。

訓練完之後會再跑一次，直接比對數字看進步多少。


In [ ]:
os.chdir("/content/PaddleOCR")

# 用『還沒訓練過』的官方 PP-OCRv4 det 模型在我們 val set 上評估
!python tools/eval.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/pretrained_models/ch_PP-OCRv4_det_student/best_accuracy \
       Eval.dataset.data_dir=/content/dataset/det/val/ \
       Eval.dataset.label_file_list=[/content/dataset/det/val_label.txt]

print("\n=== BASELINE ===")
print("請記下上面的 hmean / precision / recall，這是『沒在我們資料上訓練過』的分數")


In [ ]:
# [可選] 從 checkpoint 恢復訓練（如果斷線了）
# !python tools/train.py \
#     -c /content/det_finetune.yml \
#     -o Global.checkpoints=/content/output/det/latest

In [ ]:
os.chdir("/content/PaddleOCR")

# 訓練 log 會即時寫入 Google Drive，Colab 斷線也不會丟
# 下方畫圖 cell 就是去解析這份 log，所以跑到哪畫到哪
LOG_PATH = "/content/drive/MyDrive/ocr_project/train.log"
!mkdir -p /content/drive/MyDrive/ocr_project/

# 訓練時用 -o 覆寫 config 裡的路徑為 Colab 路徑
# 想縮短訓練（例如只跑 100 epoch）在最後加 Global.epoch_num=100 覆寫即可
# 重跑訓練會覆蓋 train.log，要保留舊的請先改檔名
!python tools/train.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/pretrained_models/ch_PP-OCRv4_det_student/best_accuracy \
       Global.save_model_dir=/content/output/det/ \
       Global.save_inference_dir=/content/output/det/inference/ \
       Train.dataset.data_dir=/content/dataset/det/train/ \
       Train.dataset.label_file_list=[/content/dataset/det/train_label.txt] \
       Eval.dataset.data_dir=/content/dataset/det/val/ \
       Eval.dataset.label_file_list=[/content/dataset/det/val_label.txt] \
    2>&1 | tee "{LOG_PATH}"


In [ ]:
# ============================================================
# AFTER TRAINING: 用 fine-tune 後的 best_accuracy 在 val 上再跑一次
# 和 Cell 15 的 baseline 直接比對 hmean，就知道進步多少
# ============================================================
!python tools/eval.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/output/det/best_accuracy \
       Eval.dataset.data_dir=/content/dataset/det/val/ \
       Eval.dataset.label_file_list=[/content/dataset/det/val_label.txt]

print("\n=== AFTER TRAINING ===")
print("和上面 baseline 的 hmean 比一下，差距就是這次 fine-tune 的效果")


## 7. 畫 Loss / Eval 曲線

解析 `train.log` 畫訓練 loss 和驗證 hmean / precision / recall 曲線。
即使 Colab 斷線只跑到一半（例如只跑了 100 epoch）也能畫，跑到哪畫到哪。

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

LOG_PATH = "/content/drive/MyDrive/ocr_project/train.log"

with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# 解析每個 training step 的 loss
loss_re = re.compile(r"epoch:\s*\[(\d+)/\d+\].*?global_step:\s*(\d+).*?loss:\s*([\d.]+)")
# 解析每次 eval 的 metric
eval_re = re.compile(r"cur metric.*?precision:\s*([\d.]+).*?recall:\s*([\d.]+).*?hmean:\s*([\d.]+)")

epochs, steps, losses = [], [], []
eval_steps, prec, rec, hmean = [], [], [], []
cur_step = 0

for line in text.splitlines():
    m = loss_re.search(line)
    if m:
        epochs.append(int(m.group(1)))
        cur_step = int(m.group(2))
        steps.append(cur_step)
        losses.append(float(m.group(3)))
        continue
    m = eval_re.search(line)
    if m:
        eval_steps.append(cur_step)
        prec.append(float(m.group(1)))
        rec.append(float(m.group(2)))
        hmean.append(float(m.group(3)))

print(f"解析到 {len(losses)} 筆 training loss、{len(hmean)} 筆 eval metric")
if epochs:
    print(f"目前跑到 epoch {epochs[-1]} / global_step {steps[-1]}")
if hmean:
    best_i = int(np.argmax(hmean))
    print(f"目前最佳 hmean = {hmean[best_i]:.4f} (在 step {eval_steps[best_i]})")

if not losses:
    print("⚠️ 還沒解析到任何 loss。確認 LOG_PATH 檔案是否存在，或訓練是否已開始。")
else:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # 左圖：訓練 loss
    ax[0].plot(steps, losses, alpha=0.3, label="raw")
    if len(losses) >= 20:
        w = max(20, len(losses) // 50)
        smooth = np.convolve(losses, np.ones(w) / w, mode="valid")
        ax[0].plot(steps[w - 1:], smooth, linewidth=2, label=f"smoothed (w={w})")
    ax[0].set_xlabel("global_step")
    ax[0].set_ylabel("loss")
    ax[0].set_title(f"Training Loss  (up to epoch {epochs[-1]})")
    ax[0].legend(); ax[0].grid(alpha=0.3)

    # 右圖：eval metrics
    if hmean:
        ax[1].plot(eval_steps, hmean, "o-", linewidth=2, label="hmean")
        ax[1].plot(eval_steps, prec, "s--", alpha=0.6, label="precision")
        ax[1].plot(eval_steps, rec, "^--", alpha=0.6, label="recall")
        ax[1].set_ylim(0, 1)
        ax[1].set_title(f"Eval Metrics  (best hmean={max(hmean):.4f})")
    else:
        ax[1].text(0.5, 0.5, "尚無 eval 資料\n(至少要跑滿一個 eval interval)",
                   ha="center", va="center", transform=ax[1].transAxes)
        ax[1].set_title("Eval Metrics")
    ax[1].set_xlabel("global_step"); ax[1].legend(); ax[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/ocr_project/train_curves.png", dpi=120)
    plt.show()
    print("\n✅ 圖已存到 /content/drive/MyDrive/ocr_project/train_curves.png")


## 8. 匯出推論模型

In [ ]:
!python tools/export_model.py \
    -c {CONFIG_PATH} \
    -o Global.pretrained_model=/content/output/det/best_accuracy \
       Global.save_inference_dir=/content/output/det/inference/

print("推論模型已匯出！")
!ls -la /content/output/det/inference/


## 9. 轉換為 ONNX

In [ ]:
!mkdir -p /content/output/det/onnx

!python -m paddle2onnx.convert \
    --model_dir /content/output/det/inference/ \
    --model_filename inference.pdmodel \
    --params_filename inference.pdiparams \
    --save_file /content/output/det/onnx/ch_PP-OCRv4_det_infer.onnx \
    --opset_version 11 \
    --enable_onnx_checker True

print("ONNX 轉換完成！")
!ls -la /content/output/det/onnx/

In [ ]:
# 驗證 ONNX 模型
import onnxruntime as ort
session = ort.InferenceSession("/content/output/det/onnx/ch_PP-OCRv4_det_infer.onnx")
for inp in session.get_inputs():
    print(f"Input:  {inp.name} shape={inp.shape} type={inp.type}")
for out in session.get_outputs():
    print(f"Output: {out.name} shape={out.shape} type={out.type}")
print("\nONNX 模型驗證通過！")

## 10. 下載模型

In [ ]:
# 方法 A：儲存到 Google Drive
!cp -r /content/output/det/ /content/drive/MyDrive/ocr_project/output_det/
print("模型已儲存到 Google Drive！")

In [ ]:
# 方法 B：直接下載 ONNX 檔案
from google.colab import files
files.download("/content/output/det/onnx/ch_PP-OCRv4_det_infer.onnx")

In [ ]:
# 方法 C：打包整個 output 下載
# !cd /content && tar czf det_output.tar.gz output/det/
# files.download("/content/det_output.tar.gz")